# Normal vs Abnormal Brain CT: DenseNet-121 trained on HDCT, tested on HDCT / LDCT / GAN

**Goal.** Train a binary classifier (normal vs abnormal) on the original high-dose CT (**HDCT**) and test whether
diagnostic performance drops on simulated low-dose CT (**LDCT**) and recovers on **GAN-enhanced** LDCT.

**Design**
- **Patient-level 5-fold cross-validation.** Patients, not slices, are split into folds, so no patient's slices appear in
  both training and test. Every patient is tested exactly once (out-of-fold).
- Inside each training fold, a separate set of **validation patients** is held out for early stopping and LR scheduling.
- The model only ever trains on HDCT. Each fold's model is then applied to the **same held-out slices** in all three
  versions (HDCT, LDCT, GAN), so the comparison is paired.
- **Patient-balanced sampling.** Each class contributes equally per batch, and within a class each patient contributes
  equally. Without this, a few patients with ~600 slices would dominate training.
- Results at **slice level** and **patient level** (patient score = mean of its slice probabilities).
- **Patient-level bootstrap 95% CIs** for the differences between versions (GAN − LDCT, etc.).

Labels are per patient (from `metaData.csv`). Every slice of an abnormal patient is labelled abnormal, even slices
where the lesion is not visible, so slice-level scores are a lower bound; patient-level scores are the clinically
meaningful ones.

> On Kaggle: **Accelerator → GPU**, **Internet → On** (to download ImageNet weights).

# Imports

In [ ]:
import os, re, glob, copy, time, random, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from concurrent.futures import ThreadPoolExecutor

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (roc_auc_score, average_precision_score, confusion_matrix,
                             roc_curve, balanced_accuracy_score, f1_score)

# Configuration

In [ ]:
SEED        = 42
N_FOLDS     = 5
PRE_SIZE    = 256      # every version is first brought to 256x256 (LDCT/GAN native size; HDCT is downsampled)
IMG_SIZE    = 224      # DenseNet input
BATCH_SIZE  = 32
NUM_EPOCHS  = 40
EPOCH_SAMPLES = 2000   # slices drawn per epoch (short epochs: with patient-balanced sampling, small patients
                       # would otherwise be repeated dozens of times per epoch and memorised)
LR_HEAD     = 1e-4     # new classifier head
LR_BACKBONE = 1e-5     # pretrained layers are only nudged
WEIGHT_DECAY = 1e-4
LABEL_SMOOTH = 0.1     # labels are per patient, so some 'abnormal' slices look normal: don't push to 0/1 extremes
LR_FACTOR   = 0.5
LR_PATIENCE = 3
ES_PATIENCE = 8
THRESHOLD   = 0.5
N_BOOT      = 2000     # bootstrap resamples for confidence intervals
QUICK_RUN   = os.environ.get('QUICK_RUN') == '1'   # tiny smoke test; leave off for real runs

DOMAINS = {'HDCT': 'HDCT', 'LDCT': 'LDCT', 'GAN': 'GAN_generated'}   # folders on disk
VERSIONS = ['HDCT', 'LDCT', 'GAN']   # what the model is tested on
COMPARED = ['LDCT', 'GAN']           # versions compared against HDCT
DOMAIN_COLORS = {'HDCT': '#2a78b5', 'LDCT': '#d1495b', 'GAN': '#2e9e6a'}

ROOT = os.environ.get('DATA_ROOT') or os.path.dirname(glob.glob('/kaggle/input/**/labels.csv', recursive=True)[0])
OUT  = os.environ.get('OUT_DIR', '/kaggle/working')
os.makedirs(OUT, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_WORKERS = 2 if DEVICE.type == 'cuda' else 0
print('Data root:', ROOT, '| Device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else '')

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
seed_everything(SEED)

# Index the slices

Folder layout: `<version>/<normal|abnormal>/<file>`. The filename prefix (e.g. `CT01`, `WR12`) is the patient ID.
The same filename exists in all three versions, so slices are paired by name.

In [ ]:
def patient_id(fname):
    return re.match(r'^(CT|WR)\d+', fname).group(0)

rows = []
for label in ['normal', 'abnormal']:
    for f in sorted(os.listdir(os.path.join(ROOT, 'HDCT', label))):
        rows.append({'file': f, 'patient': patient_id(f), 'y': int(label == 'abnormal'),
                     **{d: os.path.join(ROOT, folder, label, f) for d, folder in DOMAINS.items()}})
df = pd.DataFrame(rows)

for d, folder in DOMAINS.items():           # sanity check: identical pairing across versions
    for label in ['normal', 'abnormal']:
        assert set(os.listdir(os.path.join(ROOT, folder, label))) == set(df.loc[df.y == (label == 'abnormal'), 'file'])
assert df.groupby('patient').y.nunique().max() == 1   # one label per patient

if QUICK_RUN:   # small subset only to check the code runs end to end
    pts = df.groupby('patient').y.first()
    keep = list(pts[pts == 1].index[:8]) + list(pts[pts == 0].index[:10])
    df = df[df.patient.isin(keep)].groupby('patient').head(6).reset_index(drop=True)
    NUM_EPOCHS, N_BOOT, EPOCH_SAMPLES = 2, 50, 64

patients = df.groupby('patient').agg(y=('y', 'first'), n_slices=('file', 'size')).reset_index()
print(f"{len(df):,} slices from {len(patients)} patients")
print(patients.groupby('y').agg(patients=('patient', 'size'), slices=('n_slices', 'sum'))
      .rename(index={0: 'normal', 1: 'abnormal'}))

# Load images into memory

All three versions get identical preprocessing: grayscale → 256×256 → 224×224. HDCT is stored at raw resolution
(mostly 512×512), so it is first downsampled to 256×256, the same size as LDCT/GAN. Caching the decoded images once
avoids re-reading 36k files in every fold.

In [ ]:
def load_gray(path):
    img = Image.open(path).convert('L')
    if img.size != (PRE_SIZE, PRE_SIZE):
        img = img.resize((PRE_SIZE, PRE_SIZE), Image.BILINEAR)
    return np.asarray(img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR), dtype=np.uint8)

t0 = time.time()
IMAGES = {}
with ThreadPoolExecutor(max_workers=16) as ex:
    for d in DOMAINS:
        IMAGES[d] = np.stack(list(ex.map(load_gray, df[d])))
print({d: a.shape for d, a in IMAGES.items()}, f'loaded in {time.time() - t0:.0f}s')

# Dataset, transforms, patient-balanced sampler

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

# Geometric augmentation only. No noise / brightness / contrast augmentation: those would partly teach the model to
# cope with LDCT-like degradation and blur the HDCT vs LDCT vs GAN comparison this study is about.
train_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.95, 1.05)),
])

class SliceDataset(Dataset):
    def __init__(self, images, labels, augment=False):
        self.images, self.labels, self.augment = images, labels, augment
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        x = torch.from_numpy(self.images[i]).float().div(255).unsqueeze(0)
        if self.augment:
            x = train_aug(x)
        x = (x.expand(3, -1, -1) - MEAN) / STD
        return x, torch.tensor(self.labels[i], dtype=torch.float32)

def balance_weights(sub):
    # each class gets equal total weight; within a class each patient gets equal total weight
    n_slices = sub.groupby('patient').file.transform('size')
    n_pat = sub.groupby('y').patient.transform('nunique')
    w = 1.0 / (n_slices * n_pat)
    return (w / w.sum()).values

# Model

ImageNet-pretrained DenseNet-121. Blocks 1–3 stay frozen; only dense block 4 and the head are fine-tuned
(fewer trainable weights → less memorisation with only ~60 training patients).
The head has one output (logit for *abnormal*).

In [ ]:
def build_model(pretrained=True):
    m = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None)
    for p in m.parameters():
        p.requires_grad = False
    for name, p in m.named_parameters():
        if name.startswith(('features.denseblock4', 'features.norm5')):
            p.requires_grad = True
    m.classifier = nn.Sequential(nn.Linear(m.classifier.in_features, 512), nn.ReLU(inplace=True),
                                 nn.Dropout(0.5), nn.Linear(512, 1))
    return m.to(DEVICE)

m = build_model(pretrained=not QUICK_RUN)
print(f"Trainable params: {sum(p.numel() for p in m.parameters() if p.requires_grad):,} / "
      f"{sum(p.numel() for p in m.parameters()):,}")
del m

# Training and prediction helpers

In [ ]:
bce = nn.BCEWithLogitsLoss(reduction='none')
use_amp = DEVICE.type == 'cuda'

def run_epoch(model, loader, optimizer=None, weights=None):
    train = optimizer is not None
    model.train(train)
    tot_loss, tot_w, probs = 0.0, 0.0, []
    for b, (x, y) in enumerate(loader):
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE)
        with torch.set_grad_enabled(train), torch.autocast(DEVICE.type, enabled=use_amp):
            logit = model(x).squeeze(1)
            target = y * (1 - LABEL_SMOOTH) + 0.5 * LABEL_SMOOTH if train else y
            loss = bce(logit.float(), target)
        if train:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss.mean()).backward()
            scaler.step(optimizer); scaler.update()
            w = torch.ones_like(loss)
        else:
            w = torch.as_tensor(weights[b * loader.batch_size: b * loader.batch_size + len(y)], device=DEVICE, dtype=torch.float32)
            probs.append(torch.sigmoid(logit.float()).cpu())
        tot_loss += (loss.detach() * w).sum().item(); tot_w += w.sum().item()
    return tot_loss / tot_w, (torch.cat(probs).numpy() if probs else None)

@torch.no_grad()
def predict(model, images):
    loader = DataLoader(SliceDataset(images, np.zeros(len(images))), batch_size=BATCH_SIZE * 2,
                        shuffle=False, num_workers=NUM_WORKERS)
    model.eval(); out = []
    for x, _ in loader:
        with torch.autocast(DEVICE.type, enabled=use_amp):
            out.append(torch.sigmoid(model(x.to(DEVICE)).squeeze(1).float()).cpu())
    return torch.cat(out).numpy()

# Patient-level cross-validation

For each fold: train on HDCT (training patients) → early-stop on HDCT (validation patients) → predict the held-out
test patients in **HDCT, LDCT and GAN**.

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
df['fold'] = -1
oof = {d: np.full(len(df), np.nan) for d in VERSIONS}
history = []

for fold, (trv_idx, te_idx) in enumerate(skf.split(patients.patient, patients.y)):
    seed_everything(SEED + fold)
    trv_pat, te_pat = patients.iloc[trv_idx], patients.iloc[te_idx]
    inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED + fold)
    tr_i, va_i = next(inner.split(trv_pat.patient, trv_pat.y))
    tr_pat, va_pat = set(trv_pat.patient.iloc[tr_i]), set(trv_pat.patient.iloc[va_i])
    assert not (tr_pat & va_pat) and not ((tr_pat | va_pat) & set(te_pat.patient))   # no patient overlap

    tr_mask, va_mask, te_mask = df.patient.isin(tr_pat).values, df.patient.isin(va_pat).values, df.patient.isin(te_pat.patient).values
    df.loc[te_mask, 'fold'] = fold
    tr_df, va_df = df[tr_mask], df[va_mask]
    print(f"\n=== Fold {fold + 1}/{N_FOLDS} | patients train/val/test: {len(tr_pat)}/{len(va_pat)}/{len(te_pat)} "
          f"(abnormal {tr_df.groupby('patient').y.first().sum()}/{va_df.groupby('patient').y.first().sum()}/{te_pat.y.sum()}) "
          f"| slices: {tr_mask.sum()}/{va_mask.sum()}/{te_mask.sum()}")

    sampler = WeightedRandomSampler(torch.as_tensor(balance_weights(tr_df), dtype=torch.double),
                                    num_samples=EPOCH_SAMPLES, replacement=True)
    tr_loader = DataLoader(SliceDataset(IMAGES['HDCT'][tr_mask], tr_df.y.values, augment=True),
                           batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    va_loader = DataLoader(SliceDataset(IMAGES['HDCT'][va_mask], va_df.y.values),
                           batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    va_w = balance_weights(va_df) * len(va_df)   # patient-balanced validation loss

    model = build_model(pretrained=not QUICK_RUN)
    head = list(model.classifier.parameters())
    backbone = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith('classifier')]
    optimizer = torch.optim.AdamW([{'params': backbone, 'lr': LR_BACKBONE}, {'params': head, 'lr': LR_HEAD}],
                                  weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=LR_FACTOR,
                                                           patience=LR_PATIENCE, min_lr=1e-6)
    scaler = torch.amp.GradScaler(enabled=use_amp)
    best, best_wts, wait = np.inf, None, 0

    for epoch in range(1, NUM_EPOCHS + 1):
        t0 = time.time()
        tr_loss, _ = run_epoch(model, tr_loader, optimizer)
        va_loss, va_prob = run_epoch(model, va_loader, weights=va_w)
        va_pat_prob = pd.Series(va_prob).groupby(va_df.patient.values).mean()
        va_pat_y = va_df.groupby('patient').y.first().loc[va_pat_prob.index]
        va_auc = roc_auc_score(va_pat_y, va_pat_prob) if va_pat_y.nunique() == 2 else np.nan
        scheduler.step(va_loss)
        history.append({'fold': fold + 1, 'epoch': epoch, 'train_loss': tr_loss, 'val_loss': va_loss,
                        'val_patient_auc': va_auc, 'lr': optimizer.param_groups[1]['lr']})
        print(f"  epoch {epoch:02d}  train {tr_loss:.4f}  val {va_loss:.4f}  val patient-AUC {va_auc:.3f}  "
              f"lr(head) {optimizer.param_groups[1]['lr']:.1e}  ({time.time() - t0:.0f}s)")
        if va_loss < best:
            best, best_wts, wait = va_loss, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= ES_PATIENCE:
                print('  early stopping'); break

    model.load_state_dict(best_wts)
    torch.save({'state_dict': best_wts, 'fold': fold + 1, 'train_patients': sorted(tr_pat),
                'val_patients': sorted(va_pat), 'test_patients': sorted(te_pat.patient)},
               os.path.join(OUT, f'densenet121_fold{fold + 1}.pth'))
    test_sets = {d: IMAGES[d][te_mask] for d in VERSIONS}
    for d in VERSIONS:
        oof[d][te_mask] = predict(model, test_sets[d])
    del test_sets
    del model, optimizer; torch.cuda.empty_cache()

for d in VERSIONS:
    df[f'p_{d}'] = oof[d]
assert df[[f'p_{d}' for d in VERSIONS]].notna().all().all() and (df.fold >= 0).all()
history = pd.DataFrame(history)
df.drop(columns=list(DOMAINS)).to_csv(os.path.join(OUT, 'oof_slice_predictions.csv'), index=False)

# Training curves

In [ ]:
fig, axes = plt.subplots(1, N_FOLDS, figsize=(3.4 * N_FOLDS, 3), sharey=True)
for ax, (f, h) in zip(np.atleast_1d(axes), history.groupby('fold')):
    ax.plot(h.epoch, h.train_loss, label='train'); ax.plot(h.epoch, h.val_loss, label='val (patient-balanced)')
    ax.set_title(f'Fold {f}'); ax.set_xlabel('epoch'); ax.grid(alpha=.3)
np.atleast_1d(axes)[0].set_ylabel('BCE loss'); np.atleast_1d(axes)[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUT, 'training_curves.png'), dpi=150); plt.show()

# Metrics

*Abnormal* is the positive class. Sensitivity = abnormal correctly flagged; specificity = normal correctly cleared.
Balanced accuracy and AUC are the headline numbers because the classes are imbalanced (plain accuracy is reported too).

In [ ]:
def metrics(y, p, thr=THRESHOLD):
    y, p = np.asarray(y), np.asarray(p); pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    two = len(np.unique(y)) == 2
    return {'Accuracy': (tp + tn) / len(y), 'Balanced acc.': balanced_accuracy_score(y, pred),
            'Sensitivity': tp / (tp + fn) if tp + fn else np.nan, 'Specificity': tn / (tn + fp) if tn + fp else np.nan,
            'Precision': tp / (tp + fp) if tp + fp else np.nan, 'F1': f1_score(y, pred, zero_division=0),
            'ROC-AUC': roc_auc_score(y, p) if two else np.nan, 'PR-AUC': average_precision_score(y, p) if two else np.nan}

# patient score = mean slice probability
pat = df.groupby('patient').agg(y=('y', 'first'), fold=('fold', 'first'), n_slices=('file', 'size'),
                                **{f'p_{d}': (f'p_{d}', 'mean') for d in VERSIONS}).reset_index()
pat.to_csv(os.path.join(OUT, 'oof_patient_predictions.csv'), index=False)

summary = []
for level, data in [('slice', df), ('patient', pat)]:
    for d in VERSIONS:
        summary.append({'Level': level, 'Version': d, **metrics(data.y, data[f'p_{d}'])})
summary = pd.DataFrame(summary).set_index(['Level', 'Version'])
summary.to_csv(os.path.join(OUT, 'metrics_pooled_oof.csv'))
print('Pooled out-of-fold metrics (all patients, each tested once):')
display(summary.round(3))

# mean ± std across folds (slice level; patient level per fold has only ~19 patients)
per_fold = pd.DataFrame([{'fold': f + 1, 'Version': d, **metrics(g.y, g[f'p_{d}'])}
                         for f, g in df.groupby('fold') for d in VERSIONS])
per_fold.to_csv(os.path.join(OUT, 'metrics_per_fold_slice.csv'), index=False)
agg = per_fold.drop(columns='fold').groupby('Version', sort=False).agg(['mean', 'std'])
print('\nSlice-level, mean ± std across folds:')
display(pd.DataFrame({m: agg[m]['mean'].map('{:.3f}'.format) + ' ± ' + agg[m]['std'].map('{:.3f}'.format)
                      for m in agg.columns.get_level_values(0).unique()}, index=agg.index))

# Confusion matrices

In [ ]:
fig, axes = plt.subplots(2, len(VERSIONS), figsize=(13, 8))
for r, (level, data) in enumerate([('Slice level', df), ('Patient level', pat)]):
    for c, d in enumerate(VERSIONS):
        cm = confusion_matrix(data.y, (data[f'p_{d}'] >= THRESHOLD).astype(int), labels=[0, 1])
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[r, c],
                    xticklabels=['normal', 'abnormal'], yticklabels=['normal', 'abnormal'])
        axes[r, c].set_title(f'{d} | {level}'); axes[r, c].set_xlabel('Predicted'); axes[r, c].set_ylabel('True')
plt.tight_layout(); plt.savefig(os.path.join(OUT, 'confusion_matrices.png'), dpi=150); plt.show()

# ROC curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (level, data) in zip(axes, [('Slice level', df), ('Patient level', pat)]):
    for d in VERSIONS:
        fpr, tpr, _ = roc_curve(data.y, data[f'p_{d}'])
        ax.plot(fpr, tpr, color=DOMAIN_COLORS[d], lw=2, label=f"{d} (AUC {roc_auc_score(data.y, data[f'p_{d}']):.3f})")
    ax.plot([0, 1], [0, 1], ls='--', c='grey', lw=1)
    ax.set(title=level, xlabel='1 − specificity', ylabel='Sensitivity'); ax.legend(loc='lower right'); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(os.path.join(OUT, 'roc_curves.png'), dpi=150); plt.show()

# Metric comparison

In [ ]:
show = ['Balanced acc.', 'Sensitivity', 'Specificity', 'F1', 'ROC-AUC']
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for ax, level in zip(axes, ['slice', 'patient']):
    s = summary.loc[level, show]; x = np.arange(len(show)); w = 0.26
    for i, d in enumerate(VERSIONS):
        bars = ax.bar(x + (i - 1) * w, s.loc[d], w, label=d, color=DOMAIN_COLORS[d])
        ax.bar_label(bars, fmt='%.2f', fontsize=7, padding=1)
    ax.set_xticks(x, show); ax.set_ylim(0, 1.08); ax.set_title(f'{level.title()} level'); ax.grid(axis='y', alpha=.3)
axes[0].legend(ncol=3, loc='lower left')
plt.tight_layout(); plt.savefig(os.path.join(OUT, 'metrics_comparison.png'), dpi=150); plt.show()

# Are the differences real? Patient-level bootstrap

Patients are resampled with replacement (stratified by class, slices travel with their patient), and the difference
between two versions is recomputed each time. If the 95% CI excludes 0, the difference is unlikely to be chance.

In [ ]:
rng = np.random.default_rng(SEED)
pairs = [('GAN', 'LDCT'), ('HDCT', 'LDCT'), ('HDCT', 'GAN')]
slice_groups = {p: g.index.values for p, g in df.groupby('patient')}
pat_by_class = [pat.patient[pat.y == k].values for k in (0, 1)]
boot = {(lvl, m, a, b): [] for lvl in ('slice', 'patient') for m in ('ROC-AUC', 'Balanced acc.') for a, b in pairs}

def two_metrics(y, pa, pb):
    out = {}
    for name, fn in [('ROC-AUC', lambda yy, pp: roc_auc_score(yy, pp)),
                     ('Balanced acc.', lambda yy, pp: balanced_accuracy_score(yy, (pp >= THRESHOLD).astype(int)))]:
        out[name] = fn(y, pa) - fn(y, pb)
    return out

pat_idx = pat.set_index('patient')
for _ in range(N_BOOT):
    sample = np.concatenate([rng.choice(c, size=len(c), replace=True) for c in pat_by_class])
    sp = pat_idx.loc[sample]
    si = np.concatenate([slice_groups[p] for p in sample]); sd = df.loc[si]
    for a, b in pairs:
        for lvl, data in (('patient', sp), ('slice', sd)):
            for m, v in two_metrics(data.y.values, data[f'p_{a}'].values, data[f'p_{b}'].values).items():
                boot[(lvl, m, a, b)].append(v)

rows = []
for (lvl, m, a, b), v in boot.items():
    data = pat if lvl == 'patient' else df
    obs = two_metrics(data.y.values, data[f'p_{a}'].values, data[f'p_{b}'].values)[m]
    lo, hi = np.percentile(v, [2.5, 97.5])
    rows.append({'Level': lvl, 'Metric': m, 'Comparison': f'{a} − {b}', 'Difference': obs,
                 '95% CI low': lo, '95% CI high': hi, 'CI excludes 0': lo > 0 or hi < 0})
diffs = pd.DataFrame(rows).sort_values(['Level', 'Metric', 'Comparison'])
diffs.to_csv(os.path.join(OUT, 'bootstrap_differences.csv'), index=False)
display(diffs.round(3).reset_index(drop=True))

# Agreement with the HDCT decision (main test of the hypothesis)

The question is whether the HDCT-trained model **sees GAN images the way it sees the original HDCT images**, and LDCT
images differently. For every test slice we compare the model's output on LDCT / GAN with its output on the matching
HDCT slice. This does not depend on the ground-truth label, so it holds even where the model itself is wrong.

- **Decision agreement:** same normal/abnormal decision as on HDCT
- **Probability correlation:** Pearson r between p(version) and p(HDCT)
- **Mean |Δp|:** average absolute gap between p(version) and p(HDCT)

95% CIs come from the same patient-level bootstrap as above. `*` = the 95% CI of the difference excludes 0.

In [ ]:
def agreement_stats(data):
    out = {}
    hd_dec = data['p_HDCT'].values >= THRESHOLD
    for d in COMPARED:
        p = data[f'p_{d}'].values
        out[(d, 'Decision agreement')] = np.mean((p >= THRESHOLD) == hd_dec)
        out[(d, 'Probability correlation')] = np.corrcoef(p, data['p_HDCT'].values)[0, 1]
        out[(d, 'Mean |Δp|')] = np.mean(np.abs(p - data['p_HDCT'].values))
    return out

rng = np.random.default_rng(SEED + 1)
ag_boot = {lvl: [] for lvl in ('slice', 'patient')}
for _ in range(N_BOOT):
    sample = np.concatenate([rng.choice(c, size=len(c), replace=True) for c in pat_by_class])
    ag_boot['patient'].append(agreement_stats(pat_idx.loc[sample]))
    ag_boot['slice'].append(agreement_stats(df.loc[np.concatenate([slice_groups[p] for p in sample])]))

rows = []
for lvl, data in (('slice', df), ('patient', pat)):
    obs, bs = agreement_stats(data), pd.DataFrame(ag_boot[lvl])
    for m in ['Decision agreement', 'Probability correlation', 'Mean |Δp|']:
        row = {'Level': lvl, 'Measure': m}
        for d in COMPARED:
            lo, hi = np.percentile(bs[(d, m)], [2.5, 97.5])
            row[d] = f"{obs[(d, m)]:.3f} [{lo:.3f}, {hi:.3f}]"
        for base in ['LDCT']:
            diff = bs[('GAN', m)] - bs[(base, m)]
            lo, hi = np.percentile(diff, [2.5, 97.5])
            row[f'GAN − {base}'] = f"{obs[('GAN', m)] - obs[(base, m)]:+.3f} [{lo:+.3f}, {hi:+.3f}]" + ('  *' if lo > 0 or hi < 0 else '')
        rows.append(row)
agree = pd.DataFrame(rows)
agree.to_csv(os.path.join(OUT, 'agreement_with_hdct.csv'), index=False)
display(agree)

In [ ]:
# Probability scatter: each dot is one slice. On the diagonal = model gives the same output as on HDCT.
fig, axes = plt.subplots(1, len(COMPARED), figsize=(11, 5), sharey=True)
for ax, d in zip(axes, COMPARED):
    ax.hexbin(df['p_HDCT'], df[f'p_{d}'], gridsize=40, bins='log', cmap='viridis', mincnt=1)
    ax.plot([0, 1], [0, 1], c='white', ls='--', lw=1)
    r = np.corrcoef(df['p_HDCT'], df[f'p_{d}'])[0, 1]
    ax.set(title=f'{d} vs HDCT  (r = {r:.3f})', xlabel='p(abnormal) on HDCT', ylabel=f'p(abnormal) on {d}')
plt.tight_layout(); plt.savefig(os.path.join(OUT, 'agreement_scatter.png'), dpi=150); plt.show()

# Per-patient results

In [ ]:
labels_csv = pd.read_csv(os.path.join(ROOT, 'labels.csv'))[['patient_id', 'diagnosis']].rename(columns={'patient_id': 'patient'})
table = pat.merge(labels_csv, on='patient', how='left')
for d in VERSIONS:
    table[f'{d} correct'] = ((table[f'p_{d}'] >= THRESHOLD).astype(int) == table.y)
cols = ['patient', 'diagnosis', 'fold', 'n_slices'] + [f'p_{d}' for d in VERSIONS] + [f'{d} correct' for d in VERSIONS]
print('Abnormal patients:')
display(table[table.y == 1][cols].sort_values('n_slices', ascending=False).round(3).reset_index(drop=True))
print('Normal patients the HDCT model gets wrong:')
display(table[(table.y == 0) & ~table['HDCT correct']][cols].round(3).reset_index(drop=True))

# Outputs (in `/kaggle/working`)

| File | Contents |
|---|---|
| `densenet121_fold{1..5}.pth` | Best weights per fold + the patient IDs used for train / val / test |
| `oof_slice_predictions.csv` | Out-of-fold probability for every slice in HDCT, LDCT and GAN |
| `oof_patient_predictions.csv` | Same, aggregated per patient |
| `metrics_pooled_oof.csv`, `metrics_per_fold_slice.csv` | Metric tables |
| `bootstrap_differences.csv` | Differences between versions with 95% CIs |
| `agreement_with_hdct.csv` | How closely LDCT / GAN predictions match HDCT predictions, with 95% CIs |
| `*.png` | Figures |